# 04 — Model Training (RNN · LSTM · GRU)
**market-pulse-nn | ANN Project 1**

---

## Purpose
Train and compare three sequential deep learning models on the market-direction
classification task.  All three share the same architecture width, depth, training
loop, and evaluation protocol so that any performance differences reflect the
**inductive bias of the recurrence mechanism** — not hyperparameter differences.

| Model | Recurrence mechanism | Key property |
|-------|---------------------|--------------|
| **Vanilla RNN** | Single tanh gate | Simple; struggles with long-range dependencies (vanishing gradient) |
| **LSTM** | Input / forget / output gates + cell state | Explicitly controls what to remember and forget |
| **GRU** | Reset + update gates (no cell state) | LSTM-like performance, fewer parameters |

**Run environment:** Google Colab with **A100 GPU** (Runtime → Change runtime type → A100).
Estimated training time: ~3–4 minutes total (3 models × ~50 epochs × ~1.5 s/epoch).

---

## Setup checklist before running
1. Runtime → Change runtime type → **A100 GPU**
2. Upload the full `market-pulse-nn/` folder to your Google Drive root
3. Run cells in order — do not skip the Drive mount cell

---

## Outputs
| File | Location | Description |
|------|----------|-------------|
| `VanillaRNN_best.pt` | `saved_models/` | Best checkpoint (highest val F1) |
| `LSTM_best.pt` | `saved_models/` | Best checkpoint |
| `GRU_best.pt` | `saved_models/` | Best checkpoint |
| `all_histories.csv` | `saved_models/` | Epoch-level metrics for all 3 models |
| `training_summary.json` | `saved_models/` | Best-epoch metrics per model |
| `*.png` figures | `reports/figures/` | Training curves for IEEE report |

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
# This cell connects Colab to your Drive so we can read the sequences produced
# by Notebook 03 and save model checkpoints persistently.
# After running, your Drive appears at /content/drive/MyDrive/

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/market-pulse-nn'

# Quick sanity check — confirm the project folder is accessible
if os.path.isdir(PROJECT_ROOT):
    print(f'Project found at: {PROJECT_ROOT}')
    print(f'Contents: {os.listdir(PROJECT_ROOT)}')
else:
    raise FileNotFoundError(
        f'Project not found at {PROJECT_ROOT}. '
        'Please upload the market-pulse-nn folder to your Google Drive root.'
    )

In [ ]:
# ── Install project dependencies ─────────────────────────────────────────────
# Colab has most packages pre-installed (torch, numpy, pandas, sklearn,
# matplotlib, seaborn). We install the remaining project-specific packages.
# The -q flag suppresses verbose output.

import subprocess, sys
packages = ['ta', 'python-dotenv', 'tqdm']
print('Installing missing packages ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages,
               check=True)
print('Done.')

# Add the project root to the module search path so `from config import CFG` works
sys.path.insert(0, PROJECT_ROOT)
print(f'Added to sys.path: {PROJECT_ROOT}')

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os, json, time, warnings
from pathlib import Path
from copy import deepcopy
from dataclasses import dataclass, field
from typing import List, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score, accuracy_score

from config import CFG

warnings.filterwarnings('ignore')
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    plt.style.use('seaborn-darkgrid')

# ── Verify GPU is available ────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU  : {gpu_name}  ({gpu_mem:.1f} GB VRAM)')
else:
    print('WARNING: No GPU detected — training will be slow on CPU!')
    print('Go to Runtime → Change runtime type → GPU → A100.')

# ── Load pre-built sequences from Notebook 03 ─────────────────────────────────
npz_path = CFG.DATA_FINAL / 'sequences.npz'
data     = np.load(npz_path)

X_train, y_train = data['X_train'], data['y_train']
X_val,   y_val   = data['X_val'],   data['y_val']
X_test,  y_test  = data['X_test'],  data['y_test']

print(f'\nLoaded sequences from: {npz_path}')
print(f'  X_train : {X_train.shape}   y_train : {y_train.shape}')
print(f'  X_val   : {X_val.shape}    y_val   : {y_val.shape}')
print(f'  X_test  : {X_test.shape}    y_test  : {y_test.shape}')

N_FEATURES = X_train.shape[2]   # 27
LOOKBACK   = X_train.shape[1]   # 60
print(f'\nFeatures per timestep : {N_FEATURES}')
print(f'Sequence length       : {LOOKBACK}')

---
## Dataset & DataLoader

PyTorch's `Dataset` / `DataLoader` pipeline handles:
- Efficient batch creation
- Optional shuffling (training set only — validation and test preserve temporal order)
- Automatic GPU tensor transfer via `pin_memory`

**Batch size = 64:** Large enough for stable gradient estimates on 27-feature sequences,
small enough to fit in A100 VRAM with room to spare.

In [ ]:
class MarketDataset(Dataset):
    '''
    Wraps numpy (X, y) arrays as a PyTorch Dataset.
    Converts arrays to float32 tensors on first access — kept on CPU until
    the DataLoader moves them to the GPU in each training step.
    '''
    def __init__(self, X: np.ndarray, y: np.ndarray):
        # Store as tensors immediately — no repeated conversion during training
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def get_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test,
                    batch_size: int, num_workers: int = 2):
    '''
    Build train, val, and test DataLoaders.

    Training set is shuffled (CFG.SHUFFLE_TRAIN = True).
    Val and test sets are NOT shuffled — temporal order matters for diagnostics.
    pin_memory=True pre-pins CPU tensors for faster GPU transfer on CUDA.
    '''
    pin = (DEVICE.type == 'cuda')

    train_loader = DataLoader(
        MarketDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=CFG.SHUFFLE_TRAIN,
        num_workers=num_workers,
        pin_memory=pin,
        drop_last=True,     # drop last incomplete batch to avoid BatchNorm issues
    )
    val_loader = DataLoader(
        MarketDataset(X_val, y_val),
        batch_size=batch_size * 2,   # no gradient storage → can double batch size
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin,
    )
    test_loader = DataLoader(
        MarketDataset(X_test, y_test),
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin,
    )
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = get_dataloaders(
    X_train, y_train, X_val, y_val, X_test, y_test,
    batch_size=CFG.BATCH_SIZE,
)

print(f'Train batches : {len(train_loader):,}   '
      f'({len(train_loader.dataset):,} sequences, batch_size={CFG.BATCH_SIZE})')
print(f'Val   batches : {len(val_loader):,}')
print(f'Test  batches : {len(test_loader):,}')

---
## Model Architectures

All three models share the **same hyperparameters** to enable fair comparison:

```
Input  →  (batch, 60, 27)
           │
       [Recurrent Block: 2 layers, hidden=128, dropout=0.3]
       [VanillaRNN uses nn.RNN  |  LSTM uses nn.LSTM  |  GRU uses nn.GRU]
           │
       Take final hidden state  →  (batch, 128)
           │
       Linear(128 → 64)  →  ReLU  →  Dropout(0.3)
           │
       Linear(64 → 1)  →  raw logit
           │
Output →  (batch, 1)   [BCEWithLogitsLoss applies sigmoid internally]
```

**Why 2 layers?** A single recurrent layer may underfit the complex temporal
patterns in financial data. A second layer composes higher-order temporal
abstractions over the first layer's features.

**Why dropout=0.3?** Financial time-series models are prone to overfitting
because market regimes can shift. Dropout regularises the hidden-to-hidden
connections between recurrent layers.

In [ ]:
class VanillaRNN(nn.Module):
    '''
    Vanilla (Elman) RNN for binary sequence classification.

    Architecture:
        nn.RNN(input, hidden, num_layers, batch_first, dropout)  →  h_t
        Final hidden state h_T  →  FC(128→64)  →  ReLU  →  dropout  →  FC(64→1)

    Limitation: suffers from vanishing/exploding gradients over long sequences.
    With a 60-step window the model may fail to carry relevant signals from
    the start of the window — this is the key weakness LSTM and GRU address.
    '''
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        # batch_first=True: input shape is (batch, seq_len, features)
        # rather than the default (seq_len, batch, features)
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            nonlinearity='tanh',   # tanh is standard for Elman RNN
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        # out: (batch, seq_len, hidden_size)  — all timestep outputs
        # h_n: (num_layers, batch, hidden_size)  — final hidden states
        out, h_n = self.rnn(x)
        # Use the last layer's final hidden state as the sequence representation
        last_hidden = h_n[-1]            # (batch, hidden_size)
        return self.head(last_hidden)    # (batch, 1)


class LSTMModel(nn.Module):
    '''
    Long Short-Term Memory network for binary sequence classification.

    LSTM replaces the single tanh gate with three gating mechanisms:
      - Forget gate f_t : what fraction of the cell state to discard
      - Input gate  i_t : what new information to write into the cell state
      - Output gate o_t : what to expose to the next layer / output

    The cell state c_t acts as a long-term memory highway that passes through
    the network with only minor linear interactions — this directly combats
    the vanishing gradient problem of the Vanilla RNN.
    '''
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        # h_n: last hidden state   (num_layers, batch, hidden_size)
        # c_n: last cell state     (num_layers, batch, hidden_size)
        out, (h_n, c_n) = self.lstm(x)
        last_hidden = h_n[-1]            # (batch, hidden_size)
        return self.head(last_hidden)    # (batch, 1)


class GRUModel(nn.Module):
    '''
    Gated Recurrent Unit network for binary sequence classification.

    GRU simplifies LSTM to two gates:
      - Reset gate  r_t : how much of the past hidden state to forget
      - Update gate z_t : balance between old and new hidden state

    There is no separate cell state — the hidden state h_t serves both roles.
    GRU has ~25% fewer parameters than LSTM (no cell state projection),
    which can help generalisation when data is limited.
    '''
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, h_n = self.gru(x)
        last_hidden = h_n[-1]            # (batch, hidden_size)
        return self.head(last_hidden)    # (batch, 1)


def count_parameters(model: nn.Module) -> int:
    '''Count the number of trainable parameters in a PyTorch model.'''
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ── Instantiate all three models ──────────────────────────────────────────────
arch_kwargs = dict(
    input_size  = N_FEATURES,
    hidden_size = CFG.HIDDEN_SIZE,
    num_layers  = CFG.NUM_LAYERS,
    dropout     = CFG.DROPOUT,
)

models_registry = {
    'VanillaRNN' : VanillaRNN(**arch_kwargs),
    'LSTM'       : LSTMModel(**arch_kwargs),
    'GRU'        : GRUModel(**arch_kwargs),
}

print('Model summary:')
print(f'  {"Model":14s} {"Parameters":>12s}')
print('  ' + '-' * 28)
for name, model in models_registry.items():
    n_params = count_parameters(model)
    print(f'  {name:14s} {n_params:>12,}')

---
## Training Infrastructure

Three reusable components handle the entire training lifecycle:

| Component | Role |
|-----------|------|
| `EarlyStopping` | Monitors val F1; restores the best weights if no improvement for `patience` epochs |
| `train_one_epoch` | One forward + backward pass over the training DataLoader |
| `evaluate` | Inference-only pass over val or test DataLoader; returns loss, accuracy, F1 |
| `train_model` | Orchestrates the full training loop with scheduler + checkpointing |

**Loss function:** `BCEWithLogitsLoss`  
The model outputs a raw logit (unbounded real number). The loss function applies
sigmoid internally before computing binary cross-entropy. This is numerically more
stable than applying `sigmoid` in the model and then using `BCELoss`.

**Scheduler:** Cosine Annealing  
The learning rate follows a cosine curve from `LR_MAX` to `LR_MIN` over `T_max` epochs.
This avoids getting stuck in sharp minima and often leads to better generalisation than
a constant or step-decay schedule.

In [ ]:
class EarlyStopping:
    '''
    Monitors a validation metric and stops training when it stops improving.

    Saves the best model weights internally (deepcopy) and restores them
    when training ends — so the returned model is always the best checkpoint,
    not the last-epoch model.

    Args:
        patience  : how many consecutive epochs of no improvement before stopping
        min_delta : minimum change to count as an improvement
        mode      : 'max' for metrics like F1 / accuracy, 'min' for loss
    '''
    def __init__(self, patience: int = 15, min_delta: float = 1e-4, mode: str = 'max'):
        self.patience   = patience
        self.min_delta  = min_delta
        self.mode       = mode
        self.best_score = None
        self.best_weights = None
        self.counter    = 0
        self.stopped    = False

    def step(self, score: float, model: nn.Module) -> bool:
        '''Returns True if training should stop.'''
        improved = (
            self.best_score is None or
            (self.mode == 'max' and score > self.best_score + self.min_delta) or
            (self.mode == 'min' and score < self.best_score - self.min_delta)
        )
        if improved:
            self.best_score   = score
            self.best_weights = deepcopy(model.state_dict())
            self.counter      = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                model.load_state_dict(self.best_weights)  # restore best
                self.stopped = True
                return True
        return False

    def restore_best(self, model: nn.Module):
        '''Restore best weights after training loop ends normally.'''
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)


def train_one_epoch(model, loader, optimizer, criterion, device):
    '''
    One full pass over the training DataLoader.
    Returns (avg_loss, accuracy, f1_score) for this epoch.
    '''
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)   # GPU transfer
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)  # set_to_none is faster than zero_grad()

        logits = model(X_batch).squeeze(1)     # (batch,)
        loss   = criterion(logits, y_batch)
        loss.backward()

        # Gradient clipping: cap norm at 1.0 to prevent exploding gradients,
        # which are a known issue with Vanilla RNN on long sequences.
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item() * len(y_batch)
        preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.long().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1


@torch.no_grad()    # disables gradient tracking for faster inference & lower VRAM
def evaluate(model, loader, criterion, device):
    '''
    Inference pass over a validation or test DataLoader.
    Returns (avg_loss, accuracy, f1_score).
    @torch.no_grad() prevents PyTorch from building the computation graph,
    which saves ~50% of the memory used during training.
    '''
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        logits = model(X_batch).squeeze(1)
        loss   = criterion(logits, y_batch)

        total_loss += loss.item() * len(y_batch)
        preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.long().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1

In [ ]:
def train_model(model_name: str, model: nn.Module) -> dict:
    '''
    Complete training loop for one model.

    Orchestrates:
      - Model + optimiser + scheduler + criterion initialisation
      - Per-epoch train / validate cycle
      - EarlyStopping monitoring val F1
      - Checkpoint saving (best val F1)
      - Returns full epoch-level history as a dict of lists

    Args:
        model_name : one of 'VanillaRNN', 'LSTM', 'GRU'
        model      : uninitialised model instance (weights are reset here)

    Returns:
        history : dict with keys 'train_loss', 'val_loss', 'train_acc',
                  'val_acc', 'train_f1', 'val_f1', 'lr'
    '''
    # Move model to GPU (or CPU if no GPU available)
    model = model.to(DEVICE)

    # ── Loss function ─────────────────────────────────────────────────────────
    # BCEWithLogitsLoss = sigmoid + binary cross-entropy in one numerically stable op.
    # pos_weight > 1 can be used to penalise false negatives more heavily if
    # the dataset is imbalanced. We compute it from the training labels.
    n_pos    = int(y_train.sum())
    n_neg    = len(y_train) - n_pos
    pos_w    = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    # ── Optimiser ─────────────────────────────────────────────────────────────
    # AdamW = Adam with decoupled weight decay (L2 regularisation).
    # Decoupled means the weight decay is applied directly to the weights
    # rather than through the gradient, which is theoretically sounder.
    optimizer = AdamW(
        model.parameters(),
        lr=CFG.LEARNING_RATE,
        weight_decay=CFG.WEIGHT_DECAY,
    )

    # ── Learning rate scheduler ───────────────────────────────────────────────
    # Cosine annealing decays LR from initial value to eta_min over T_max epochs.
    # The smooth decay avoids oscillating around minima and typically finds
    # flatter, better-generalising solutions than step decay.
    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=CFG.EPOCHS,
        eta_min=CFG.LEARNING_RATE * 0.01,  # floor = 1% of initial LR
    )

    # ── Early stopping ────────────────────────────────────────────────────────
    early_stop = EarlyStopping(patience=CFG.PATIENCE, mode='max')

    # ── History tracking ──────────────────────────────────────────────────────
    history = {k: [] for k in ['train_loss', 'val_loss', 'train_acc', 'val_acc',
                                'train_f1',  'val_f1',  'lr']}

    print(f'\n{'='*60}')
    print(f'  Training {model_name}')
    print(f'  Params    : {count_parameters(model):,}')
    print(f'  Epochs    : {CFG.EPOCHS}   Patience: {CFG.PATIENCE}')
    print(f'  LR        : {CFG.LEARNING_RATE}  → {CFG.LEARNING_RATE*0.01} (cosine)')
    print(f'  pos_weight: {pos_w.item():.3f}')
    print(f'{'='*60}')

    best_val_f1 = 0.0
    start_time  = time.time()

    for epoch in tqdm(range(1, CFG.EPOCHS + 1), desc=model_name, ncols=80):
        # ── One training epoch ────────────────────────────────────────────────
        tr_loss, tr_acc, tr_f1 = train_one_epoch(
            model, train_loader, optimizer, criterion, DEVICE
        )
        # ── Validation pass ───────────────────────────────────────────────────
        vl_loss, vl_acc, vl_f1 = evaluate(
            model, val_loader, criterion, DEVICE
        )
        # ── Scheduler step ────────────────────────────────────────────────────
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        # ── Record metrics ────────────────────────────────────────────────────
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        history['train_f1'].append(tr_f1)
        history['val_f1'].append(vl_f1)
        history['lr'].append(current_lr)

        # ── Checkpoint: save if best val F1 so far ────────────────────────────
        if vl_f1 > best_val_f1 and CFG.SAVE_BEST:
            best_val_f1 = vl_f1
            ckpt_path   = CFG.MODELS / f'{model_name}_best.pt'
            torch.save({
                'epoch'      : epoch,
                'model_state': model.state_dict(),
                'val_f1'     : vl_f1,
                'val_acc'    : vl_acc,
                'val_loss'   : vl_loss,
            }, ckpt_path)

        # ── Early stopping check ──────────────────────────────────────────────
        if early_stop.step(vl_f1, model):
            tqdm.write(f'  Early stop at epoch {epoch}. Best val F1={early_stop.best_score:.4f}')
            break

    # Restore best weights after loop ends (whether by early stop or max epochs)
    early_stop.restore_best(model)

    elapsed = time.time() - start_time
    best_ep = int(np.argmax(history['val_f1'])) + 1
    print(f'\n  Done in {elapsed/60:.1f} min')
    print(f'  Best epoch : {best_ep}  |  val F1={max(history["val_f1"]):.4f}  |  '
          f'val Acc={history["val_acc"][best_ep-1]:.4f}')

    return history

---
## Training

Each model is trained with identical hyperparameters. After each model,
a live loss + F1 curve is printed to monitor convergence.

Expected behaviour:
- **VanillaRNN** — converges fastest but plateaus earliest. May show train/val gap due to gradient issues.
- **LSTM** — converges more slowly but typically reaches better val F1.
- **GRU** — similar to LSTM, occasionally better due to fewer parameters reducing overfitting.

In [ ]:
# ── Train Vanilla RNN ────────────────────────────────────────────────────────
# Re-instantiate to ensure fresh random initialisation (Xavier uniform by default
# in nn.RNN). We do not share weights between models.
rnn_model   = VanillaRNN(**arch_kwargs)
rnn_history = train_model('VanillaRNN', rnn_model)

# ── Quick curve after training ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, len(rnn_history['train_loss']) + 1)

axes[0].plot(ep, rnn_history['train_loss'], label='Train', lw=1.5)
axes[0].plot(ep, rnn_history['val_loss'],   label='Val',   lw=1.5, ls='--')
axes[0].set_title('VanillaRNN — Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, rnn_history['train_f1'], label='Train F1', lw=1.5)
axes[1].plot(ep, rnn_history['val_f1'],   label='Val F1',   lw=1.5, ls='--')
best_ep = int(np.argmax(rnn_history['val_f1'])) + 1
axes[1].axvline(best_ep, color='red', lw=1.2, ls=':', label=f'Best epoch ({best_ep})')
axes[1].set_title('VanillaRNN — Macro F1', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 Score')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Train LSTM ───────────────────────────────────────────────────────────────
lstm_model   = LSTMModel(**arch_kwargs)
lstm_history = train_model('LSTM', lstm_model)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, len(lstm_history['train_loss']) + 1)

axes[0].plot(ep, lstm_history['train_loss'], label='Train', lw=1.5, color='#FF9800')
axes[0].plot(ep, lstm_history['val_loss'],   label='Val',   lw=1.5, ls='--', color='#FF9800')
axes[0].set_title('LSTM — Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, lstm_history['train_f1'], label='Train F1', lw=1.5, color='#FF9800')
axes[1].plot(ep, lstm_history['val_f1'],   label='Val F1',   lw=1.5, ls='--', color='#FF9800')
best_ep = int(np.argmax(lstm_history['val_f1'])) + 1
axes[1].axvline(best_ep, color='red', lw=1.2, ls=':', label=f'Best epoch ({best_ep})')
axes[1].set_title('LSTM — Macro F1', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 Score')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Train GRU ────────────────────────────────────────────────────────────────
gru_model   = GRUModel(**arch_kwargs)
gru_history = train_model('GRU', gru_model)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, len(gru_history['train_loss']) + 1)

axes[0].plot(ep, gru_history['train_loss'], label='Train', lw=1.5, color='#4CAF50')
axes[0].plot(ep, gru_history['val_loss'],   label='Val',   lw=1.5, ls='--', color='#4CAF50')
axes[0].set_title('GRU — Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, gru_history['train_f1'], label='Train F1', lw=1.5, color='#4CAF50')
axes[1].plot(ep, gru_history['val_f1'],   label='Val F1',   lw=1.5, ls='--', color='#4CAF50')
best_ep = int(np.argmax(gru_history['val_f1'])) + 1
axes[1].axvline(best_ep, color='red', lw=1.2, ls=':', label=f'Best epoch ({best_ep})')
axes[1].set_title('GRU — Macro F1', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 Score')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Results Analysis

The comparison chart below overlays all three models on the same axes —
this is the primary figure for the IEEE report's **Results** section.

What to look for:
- **Val loss curve shape:** monotone decrease with plateau = healthy training
- **Train/val gap:** large gap = overfitting; use this to justify dropout values
- **Best val F1 epoch:** earlier = simpler model; later = more complex loss landscape
- **Final val F1 ranking:** the key number for the comparison table

In [ ]:
# ── Combined training curves — all 3 models on the same axes ────────────────
# This 2×2 grid is the key figure for the IEEE Results section.
# Layout: [Loss | F1] on top row, [Accuracy | LR schedule] on bottom row.

histories = {
    'VanillaRNN' : rnn_history,
    'LSTM'       : lstm_history,
    'GRU'        : gru_history,
}
colors = {'VanillaRNN': '#2196F3', 'LSTM': '#FF9800', 'GRU': '#4CAF50'}
styles = {'train': '-', 'val': '--'}

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle('Training Curves — All Models Comparison  (Colab A100)',
             fontsize=15, fontweight='bold', y=1.01)

metrics = [
    ('train_loss', 'val_loss',  'BCE Loss',    'Training & Validation Loss',      axes[0, 0]),
    ('train_f1',   'val_f1',    'Macro F1',    'Training & Validation F1 Score',  axes[0, 1]),
    ('train_acc',  'val_acc',   'Accuracy',    'Training & Validation Accuracy',  axes[1, 0]),
]

for tr_key, vl_key, ylabel, title, ax in metrics:
    for model_name, hist in histories.items():
        ep = range(1, len(hist[tr_key]) + 1)
        ax.plot(ep, hist[tr_key], lw=1.8, color=colors[model_name],
                label=f'{model_name} train', alpha=0.9)
        ax.plot(ep, hist[vl_key], lw=1.8, color=colors[model_name],
                ls='--', label=f'{model_name} val', alpha=0.9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=8, ncol=2, framealpha=0.85)
    ax.grid(True, alpha=0.25)

# ── Bottom right: learning rate schedule ─────────────────────────────────────
# Shows the cosine annealing curve. All models use the same schedule,
# but may stop at different epochs (early stopping), so the curves differ in length.
ax_lr = axes[1, 1]
for model_name, hist in histories.items():
    ep = range(1, len(hist['lr']) + 1)
    ax_lr.plot(ep, hist['lr'], lw=1.8, color=colors[model_name], label=model_name)
ax_lr.set_title('Learning Rate Schedule (Cosine Annealing)', fontsize=12, fontweight='bold')
ax_lr.set_xlabel('Epoch', fontsize=10)
ax_lr.set_ylabel('Learning Rate', fontsize=10)
ax_lr.legend(fontsize=9, framealpha=0.85)
ax_lr.grid(True, alpha=0.25)
ax_lr.set_yscale('log')   # log scale makes the cosine shape more visible

plt.tight_layout()
fig.savefig(CFG.FIGURES / '08_training_curves_all_models.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/08_training_curves_all_models.png')

In [ ]:
# ── Best-epoch metrics table ─────────────────────────────────────────────────
# For each model, pick the epoch with the highest val F1 and report all metrics.
# This is the table that goes into the IEEE report's Results section.

print(f'\n{'Model':14s}  {'Best Ep':>7s}  {'Train Loss':>10s}  {'Val Loss':>10s}  '
      f'{'Train Acc':>10s}  {'Val Acc':>10s}  {'Train F1':>9s}  {'Val F1':>9s}  {'Params':>10s}')
print('-' * 110)

summary = {}
for model_name, hist in histories.items():
    best_ep   = int(np.argmax(hist['val_f1']))
    n_params  = count_parameters(models_registry[model_name])
    row = {
        'best_epoch' : best_ep + 1,
        'train_loss' : hist['train_loss'][best_ep],
        'val_loss'   : hist['val_loss'][best_ep],
        'train_acc'  : hist['train_acc'][best_ep],
        'val_acc'    : hist['val_acc'][best_ep],
        'train_f1'   : hist['train_f1'][best_ep],
        'val_f1'     : hist['val_f1'][best_ep],
        'n_params'   : n_params,
    }
    summary[model_name] = row
    print(f'{model_name:14s}  {row["best_epoch"]:>7d}  {row["train_loss"]:>10.4f}  '
          f'{row["val_loss"]:>10.4f}  {row["train_acc"]:>10.4f}  {row["val_acc"]:>10.4f}  '
          f'{row["train_f1"]:>9.4f}  {row["val_f1"]:>9.4f}  {row["n_params"]:>10,}')

print()
best_model = max(summary, key=lambda k: summary[k]['val_f1'])
print(f'Best model by val F1: {best_model}  (F1={summary[best_model]["val_f1"]:.4f})')

In [ ]:
# ── Best val F1 bar chart — clean summary figure for the report ──────────────
fig, ax = plt.subplots(figsize=(8, 5))

model_names = list(summary.keys())
val_f1s     = [summary[m]['val_f1'] for m in model_names]
val_accs    = [summary[m]['val_acc'] for m in model_names]
bar_colors  = [colors[m] for m in model_names]

x = np.arange(len(model_names))
w = 0.38
b1 = ax.bar(x - w/2, val_f1s,  w, label='Val Macro F1',  color=bar_colors,
             alpha=0.9, edgecolor='white', lw=1.5)
b2 = ax.bar(x + w/2, val_accs, w, label='Val Accuracy',  color=bar_colors,
             alpha=0.5, edgecolor='white', lw=1.5, hatch='//')

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.4f}', ha='center', va='bottom',
            fontsize=10, fontweight='bold')
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_title('Best Val F1 & Accuracy — Model Comparison', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=11)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
fig.savefig(CFG.FIGURES / '09_model_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/09_model_comparison_bar.png')

In [ ]:
# ── Save training histories and summary for Notebook 05 ─────────────────────
#
# all_histories.csv  — full epoch-level table: (epoch, model, metric, value)
#   Used in NB05 to reproduce training curves without re-training.
#
# training_summary.json — best-epoch metrics per model.
#   Used in NB05 for the final evaluation table and IEEE report.

# ── all_histories.csv ─────────────────────────────────────────────────────────
rows = []
for model_name, hist in histories.items():
    for epoch_idx, _ in enumerate(hist['train_loss']):
        row = {'model': model_name, 'epoch': epoch_idx + 1}
        for metric in ['train_loss', 'val_loss', 'train_acc', 'val_acc',
                       'train_f1', 'val_f1', 'lr']:
            row[metric] = hist[metric][epoch_idx]
        rows.append(row)

hist_df   = pd.DataFrame(rows)
hist_path = CFG.MODELS / 'all_histories.csv'
hist_df.to_csv(hist_path, index=False)
print(f'[1/2]  all_histories.csv   saved  ({len(hist_df):,} rows)  →  {hist_path}')

# ── training_summary.json ─────────────────────────────────────────────────────
json_path = CFG.MODELS / 'training_summary.json'
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'[2/2]  training_summary.json saved               →  {json_path}')

# ── List all saved checkpoints ────────────────────────────────────────────────
print('\nSaved model checkpoints:')
for pt_file in sorted(CFG.MODELS.glob('*.pt')):
    size_mb = pt_file.stat().st_size / 1e6
    print(f'  {pt_file.name:30s}  {size_mb:.2f} MB')

In [ ]:
# ── Final notebook summary ───────────────────────────────────────────────────
print('=' * 65)
print('  Notebook 04 — Model Training')
print('  STATUS: COMPLETE')
print('=' * 65)
print()
print(f'  GPU used     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'  Models trained: {list(histories.keys())}')
print()
print('  Best-epoch validation results:')
print(f'  {"Model":14s}  {"Val Acc":>9s}  {"Val F1":>9s}')
print('  ' + '-' * 38)
for m, r in summary.items():
    marker = ' ← best' if m == best_model else ''
    print(f'  {m:14s}  {r["val_acc"]:>9.4f}  {r["val_f1"]:>9.4f}{marker}')
print()
print('  Checkpoints saved to saved_models/:')
for m in histories:
    print(f'    {m}_best.pt')
print('  Histories: all_histories.csv')
print('  Summary  : training_summary.json')
print()
print('  Next: Run 05_evaluation.ipynb')
print('        Load best checkpoints → confusion matrix, ROC, PR curves,')
print('        RMSE on probabilities, final test-set comparison table.')
print('=' * 65)